# Baseline Models Training
This notebook trains the baseline models on the processed UNSW-NB15 dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

from src.models.baselines import RandomForestBaseline, XGBoostBaseline, MLPBaseline, evaluate_binary


In [ ]:
PROJECT_ROOT = Path(os.getcwd()).parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = RESULTS_DIR / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Try loading data
try:
    X_train = np.load(PROCESSED_DIR / "X_train.npy")
    X_test = np.load(PROCESSED_DIR / "X_test.npy")
    y_train = np.load(PROCESSED_DIR / "y_train.npy")
    y_test = np.load(PROCESSED_DIR / "y_test.npy")
    print(f"Data loaded: X_train {X_train.shape}, X_test {X_test.shape}")
except FileNotFoundError:
    print("Error: Processed data not found. Please run preprocessing pipeline first.")


In [ ]:
# Initialize models
models = [
    RandomForestBaseline(),
    XGBoostBaseline(),
    MLPBaseline()
]

results = []
for model in models:
    print(f"\nTraining {model.name}...")
    try:
        model.fit(X_train, y_train)
        model.save(MODELS_DIR / f"{model.name}.model")
        
        # Evaluate
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        metrics = evaluate_binary(y_test, y_pred, y_proba, model_name=model.name)
        results.append(metrics)
        
        # Show cm
        print(f"{model.name} Confusion Matrix:\n", metrics["confusion_matrix"])
    except NameError:
        print("Data not loaded, skipping training.")


In [ ]:
# Generate comparison table
if results:
    summary = []
    for res in results:
        summary.append({
            "Model": res["model"],
            "Accuracy": res["accuracy"],
            "Precision": res["precision_macro"],
            "Recall": res["recall_macro"],
            "F1": res["f1_macro"]
        })
    df_summary = pd.DataFrame(summary)
    
    # Save CSV
    df_summary.to_csv(RESULTS_DIR / "baseline_comparison.csv", index=False)
    # Save Markdown
    df_summary.to_markdown(RESULTS_DIR / "baseline_comparison.md", index=False)
    
    display(df_summary)
